# Group comparisons (primary analysis)

Compares patients, healthcare professionals and AI developers on the 12 primary outcomes.

- One test per outcome: chi-square for the yes/no and three-answer outcomes, Kruskal-Wallis for the seven ordered outcomes. Fisher's exact test is used instead of chi-square if expected counts are small.
- Holm correction across the 12 tests.
- Effect sizes (Cramer's V, epsilon-squared) with 95% bootstrap intervals; the bootstrap resamples within each group, so group sizes stay fixed.
- Pairwise comparisons only for outcomes that differ after Holm correction.

**Reads:** `data/survey_clean.csv`

**Writes to `results/`:** `primary_tests.csv`, `pairwise.csv`

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA_FILE = Path("../data/survey_clean.csv")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

GROUPS = ["Patient", "Healthcare professional", "AI developer"]

# Answer options in the order used in the analysis.
# For Q7, Q9, Q14 and Q16 this is meaning order (Not < Neutral < Somewhat < ... < Extremely);
# on the form, Neutral was the third option.
CATEGORIES = {
    "gender": ["Male", "Female", "Non-binary", "Prefer not to say"],
    "age_group": ["18-24", "25-34", "35-44", "45-54", "55-64", "65 and above"],
    "experience": ["Less than 1 year", "1-3 years", "4-6 years", "7-10 years", "More than 10 years"],
    "familiarity": ["Not familiar at all", "Somewhat familiar", "Moderately familiar", "Familiar", "Very familiar"],
    "bias_concern": ["Not concerned", "Neutral", "Somewhat concerned", "Concerned", "Extremely concerned"],
    "privacy_protection": ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"],
    "explanation_importance": ["Not important", "Neutral", "Somewhat important", "Important", "Extremely important"],
    "inform_patients": ["Yes", "No", "Not sure"],
    "trust": ["No trust", "Little trust", "Complete trust"],
    "autonomy_undermined": ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"],
    "guidelines_confidence": ["Not confident", "Neutral", "Somewhat confident", "Confident", "Extremely confident"],
    "no_human_life_death": ["Yes", "No", "Not sure"],
    "disparities_concern": ["Not concerned", "Neutral", "Somewhat concerned", "Concerned", "Extremely concerned"],
    "continent": ["Africa", "Asia", "Europe", "North America", "South America", "Oceania (Australia and the Pacific)"],
}

SEED = 2026
N_BOOT = 2000           # bootstrap resamples for effect size intervals
N_MONTE_CARLO = 10000   # resamples for Fisher's exact test on tables larger than 2 x 2
ALPHA = 0.05

OUTCOMES = [  # (number, label, column, type)
    (1,  "Q7 bias concern",                          "bias_concern_code",           "ordered"),
    (2,  "Q8 privacy is protected",                  "privacy_protection_code",     "ordered"),
    (3,  "Q9 explanations important",               "explanation_importance_code", "ordered"),
    (4,  "Q10 inform patients",                      "inform_patients",             "categories"),
    (5,  "Q11 accountable: AI developers",           "acc_ai_developers",           "binary"),
    (6,  "Q11 accountable: healthcare professional", "acc_healthcare_professional", "binary"),
    (7,  "Q11 accountable: hospital or clinic",      "acc_hospital_clinic",         "binary"),
    (8,  "Q12 trust",                                "trust_code",                  "ordered"),
    (9,  "Q13 autonomy could be undermined",         "autonomy_undermined_code",    "ordered"),
    (10, "Q14 guidelines are enough",                "guidelines_confidence_code",  "ordered"),
    (11, "Q15 comfortable with no human",            "no_human_life_death",         "categories"),
    (12, "Q16 disparities concern",                  "disparities_concern_code",    "ordered"),
]

## Load the data

In [2]:
df = pd.read_csv(DATA_FILE)
for var, cats in CATEGORIES.items():
    assert set(df[var].dropna()) <= set(cats), var
    df[var] = pd.Categorical(df[var], categories=cats, ordered=True)
df["group"] = pd.Categorical(df["group"], categories=GROUPS)
for _, _, var, _ in OUTCOMES:
    assert df[var].notna().all(), var
print("Participants:", len(df))
df["group"].value_counts(sort=False)

Participants: 253


group
Patient                     48
Healthcare professional     58
AI developer               147
Name: count, dtype: int64

## Test functions

In [3]:
rng = np.random.default_rng(SEED)


def crosstab(data, var):
    return pd.crosstab(data[var], data["group"]).reindex(columns=GROUPS, fill_value=0)


def expected_ok(table):
    """Chi-square is used if no more than 20% of expected counts are below 5 and none is below 1."""
    expected = stats.contingency.expected_freq(table.values)
    return (expected < 5).mean() <= 0.20 and (expected >= 1).all(), expected.min()


def cramers_v(table):
    t = table.values
    chi2 = stats.chi2_contingency(t, correction=False)[0]
    k = min(t.shape) - 1
    return float(np.sqrt(chi2 / (t.sum() * k))) if k > 0 else np.nan


def epsilon_squared(data, var):
    h = stats.kruskal(*[data.loc[data["group"] == g, var] for g in GROUPS]).statistic
    return float(h / (len(data) - 1))


def table_test(table):
    ok, min_exp = expected_ok(table)
    if ok:
        res = stats.chi2_contingency(table.values, correction=False)
        return "Chi-square", float(res.statistic), int(res.dof), float(res.pvalue), min_exp
    if table.shape == (2, 2):
        res = stats.fisher_exact(table.values)
    else:
        res = stats.fisher_exact(table.values, method=stats.MonteCarloMethod(n_resamples=N_MONTE_CARLO, rng=rng))
    return "Fisher exact", np.nan, np.nan, float(res.pvalue), min_exp


def boot_ci(data, func):
    """95% bootstrap interval, resampling within each group."""
    parts = [data[data["group"] == g] for g in GROUPS]
    values = [func(pd.concat([p.iloc[rng.integers(0, len(p), len(p))] for p in parts])) for _ in range(N_BOOT)]
    return float(np.nanpercentile(values, 2.5)), float(np.nanpercentile(values, 97.5))

## The 12 primary tests

In [4]:
rows = []
for num, label, var, kind in OUTCOMES:
    row = {"outcome_no": num, "outcome": label, "variable": var, "type": kind}
    if kind in ("binary", "categories"):
        table = crosstab(df, var)
        test, stat, dof, p, min_exp = table_test(table)
        lo, hi = boot_ci(df, lambda d: cramers_v(crosstab(d, var)))
        row.update(test=test, statistic=stat, df=dof, p=p, min_expected=round(min_exp, 1),
                   effect_size_name="Cramer's V", effect_size=cramers_v(table), es_ci_low=lo, es_ci_high=hi)
        if kind == "binary":
            for g in GROUPS:
                row["pct_" + g] = round(100 * df.loc[df["group"] == g, var].mean(), 1)
    else:
        res = stats.kruskal(*[df.loc[df["group"] == g, var] for g in GROUPS])
        lo, hi = boot_ci(df, lambda d: epsilon_squared(d, var))
        row.update(test="Kruskal-Wallis", statistic=float(res.statistic), df=2, p=float(res.pvalue),
                   min_expected=np.nan, effect_size_name="Epsilon-squared", effect_size=epsilon_squared(df, var),
                   es_ci_low=lo, es_ci_high=hi)
        ranks = stats.rankdata(df[var])
        for g in GROUPS:
            row["mean_rank_" + g] = round(float(ranks[df["group"] == g].mean()), 1)
    rows.append(row)

primary = pd.DataFrame(rows)
primary["p_holm"] = multipletests(primary["p"], method="holm")[1]
primary["difference_after_holm"] = primary["p_holm"] < ALPHA
primary.to_csv(RESULTS / "primary_tests.csv", index=False)

print("Outcomes with a group difference after Holm correction:", int(primary["difference_after_holm"].sum()), "of 12")
primary[["outcome_no", "outcome", "test", "statistic", "df", "p", "p_holm", "effect_size_name",
         "effect_size", "es_ci_low", "es_ci_high"]].round(4)

Outcomes with a group difference after Holm correction: 0 of 12


,outcome_no,outcome,test,statistic,df,p,p_holm,effect_size_name,effect_size,es_ci_low,es_ci_high
0,1,Q7 bias concern,Kruskal-Wallis,0.3415,2,0.8431,1.0000,Epsilon-squared,0.0014,0.0003,0.0357
1,2,Q8 privacy is protected,Kruskal-Wallis,0.8257,2,0.6618,1.0000,Epsilon-squared,0.0033,0.0003,0.0382
2,3,Q9 explanations important,Kruskal-Wallis,5.9184,2,0.0519,0.5705,Epsilon-squared,0.0235,0.0030,0.0752
3,4,Q10 inform patients,Chi-square,3.8316,4,0.4293,1.0000,Cramer's V,0.0870,0.0514,0.1990
4,5,Q11 accountable: AI developers,Chi-square,8.4571,2,0.0146,0.1749,Cramer's V,0.1828,0.0732,0.3180
5,6,Q11 accountable: healthcare professional,Chi-square,0.7996,2,0.6704,1.0000,Cramer's V,0.0562,0.0176,0.1980
6,7,Q11 accountable: hospital or clinic,Chi-square,3.8946,2,0.1427,1.0000,Cramer's V,0.1241,0.0339,0.2583
7,8,Q12 trust,Kruskal-Wallis,0.0388,2,0.9808,1.0000,Epsilon-squared,0.0002,0.0002,0.0266
8,9,Q13 autonomy could be undermined,Kruskal-Wallis,0.0716,2,0.9648,1.0000,Epsilon-squared,0.0003,0.0002,0.0291
9,10,Q14 guidelines are enough,Kruskal-Wallis,2.0965,2,0.3505,1.0000,Epsilon-squared,0.0083,0.0007,0.0527


Cramer's V and epsilon-squared cannot be below zero, so a lower limit above zero does not by itself mean there is a difference; use the Holm-corrected p-value for that. The upper limit shows the largest group difference that fits the data.

## Pairwise comparisons

Run only for outcomes that differ after Holm correction: odds ratios (yes/no), Cramer's V (three answers) or Dunn's test with rank-biserial correlation (ordered), with Holm correction within each outcome.

In [5]:
def odds_ratio_ci(table):
    """Odds ratio (first group vs second) with 95% CI; 0.5 is added to all cells if any cell is zero."""
    a, b = table.loc[1].values if 1 in table.index else (0, 0)
    c, d = table.loc[0].values if 0 in table.index else (0, 0)
    if min(a, b, c, d) == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    or_ = (a * d) / (b * c)
    se = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    return or_, np.exp(np.log(or_) - 1.96 * se), np.exp(np.log(or_) + 1.96 * se)


def dunn(data, var):
    ranks = stats.rankdata(data[var])
    n = len(data)
    _, ties = np.unique(data[var], return_counts=True)
    tie_term = (ties ** 3 - ties).sum() / (12 * (n - 1))
    out = []
    for g1, g2 in combinations(GROUPS, 2):
        m1, m2 = data["group"] == g1, data["group"] == g2
        z = (ranks[m1].mean() - ranks[m2].mean()) / np.sqrt((n * (n + 1) / 12 - tie_term) * (1 / m1.sum() + 1 / m2.sum()))
        u = stats.mannwhitneyu(data.loc[m1, var], data.loc[m2, var]).statistic
        out.append((g1, g2, z, 2 * stats.norm.sf(abs(z)), 2 * u / (m1.sum() * m2.sum()) - 1))
    return out


rows = []
for _, r in primary[primary["difference_after_holm"]].iterrows():
    var, kind = r["variable"], r["type"]
    if kind == "ordered":
        for g1, g2, z, p, rbc in dunn(df, var):
            rows.append({"outcome": r["outcome"], "pair": g1 + " vs " + g2, "test": "Dunn", "statistic": z, "p": p,
                         "effect_size_name": "rank-biserial", "effect_size": rbc})
    else:
        for g1, g2 in combinations(GROUPS, 2):
            sub = df[df["group"].isin([g1, g2])]
            table = pd.crosstab(sub[var], sub["group"]).reindex(columns=[g1, g2], fill_value=0)
            test, stat, _, p, _ = table_test(table)
            entry = {"outcome": r["outcome"], "pair": g1 + " vs " + g2, "test": test, "statistic": stat, "p": p}
            if kind == "binary":
                or_, lo, hi = odds_ratio_ci(table)
                entry.update(effect_size_name="odds ratio (first vs second)", effect_size=or_, es_ci_low=lo, es_ci_high=hi)
            else:
                entry.update(effect_size_name="Cramer's V", effect_size=cramers_v(table))
            rows.append(entry)

pairwise = pd.DataFrame(rows, columns=["outcome", "pair", "test", "statistic", "p", "effect_size_name",
                                       "effect_size", "es_ci_low", "es_ci_high"])
pairwise["p_holm_within_outcome"] = (pairwise.groupby("outcome")["p"].transform(lambda s: multipletests(s, method="holm")[1])
                                     if len(pairwise) else [])
pairwise.to_csv(RESULTS / "pairwise.csv", index=False)
print("Pairwise comparisons run for:", sorted(pairwise["outcome"].unique()) if len(pairwise) else "none (no outcome differed after Holm)")

Pairwise comparisons run for: none (no outcome differed after Holm)
